# Phase 3 Kalman baseline review

Run a Phase 3 EMR backtest first. This notebook evaluates the **existing Kalman baseline only**. It does not train or score an ML model. The selected cases are report-anchored DMV storm episodes, so these metrics describe performance on storm-event windows—not all weather.

In [ ]:
%pip install -e "../.[nexrad]"

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo = Path.cwd().parent
# Use whichever completed Phase 3 result was downloaded most recently.
result_candidates = [
    repo / 'data' / 'phase3_latest_kalman_results.jsonl',
    repo / 'data' / 'phase3_2024_kalman_results.jsonl',
]
available_results = [path for path in result_candidates if path.exists()]
if not available_results:
    raise RuntimeError('Run a Phase 3 EMR command first, then download its results.')
results_path = max(available_results, key=lambda path: path.stat().st_mtime)

results = pd.read_json(results_path, lines=True)
for column in ['scan_time', 'forecast_time']:
    results[column] = pd.to_datetime(results[column], utc=True)
print(f'Loaded {len(results):,} forecast/observation rows from {results.event_id.nunique():,} storm episodes.')

## Baseline scorecard

- **Observed hit rate** is how often ≥35 dBZ actually covered the 2 km McLean target.
- **Mean forecast probability** is what the baseline predicted on average. It should be compared with the observed hit rate, but not treated as a calibration verdict alone.
- **Brier score** is mean squared probability error; lower is better. It is only meaningful when comparing the same data split and horizon with another model.

In [ ]:
def score_horizon(frame):
    probability = frame['combined_probability'].astype(float)
    observed = frame['observed_35dbz_coverage'].astype(float)
    return pd.Series({
        'forecast_rows': len(frame),
        'storm_episodes': frame['event_id'].nunique(),
        'observed_hit_rate': observed.mean(),
        'mean_forecast_probability': probability.mean(),
        'brier_score_lower_is_better': ((probability - observed) ** 2).mean(),
        'mean_active_tracks': frame['active_track_count'].mean(),
    })

scorecard = results.groupby('horizon_minutes').apply(score_horizon, include_groups=False)
display(scorecard.style.format({
    'observed_hit_rate': '{:.1%}', 'mean_forecast_probability': '{:.1%}',
    'brier_score_lower_is_better': '{:.4f}', 'mean_active_tracks': '{:.1f}',
}))

## Did the model make the right office-impact call?

This is the plain-English result. We call it a **predicted hit** when the model gives McLean at least a 5% chance of ≥35 dBZ weather within the stated window. The rows show what the model said; the columns show what later radar actually observed.

- **Correct hit:** the model alerted and weather reached the office.
- **False alarm:** the model alerted but weather did not reach the office.
- **Miss:** the model did not alert but weather reached the office.
- **Correct no-hit:** the model did not alert and weather did not reach the office.

The 5% cutoff is an early-warning policy for this still-uncalibrated baseline—not a scientifically final threshold. Change it in the next cell to explore the trade-off.

In [ ]:
ALERT_THRESHOLD = 0.05  # Say 'likely to hit' at 5% or higher. Try 0.10 for fewer alerts.

summaries, matrices = [], {}
for horizon, frame in results.groupby('horizon_minutes'):
    predicted_hit = frame['combined_probability'].astype(float).ge(ALERT_THRESHOLD)
    observed_hit = frame['observed_35dbz_coverage'].astype(bool)
    correct_hits = int((predicted_hit & observed_hit).sum())
    false_alarms = int((predicted_hit & ~observed_hit).sum())
    misses = int((~predicted_hit & observed_hit).sum())
    correct_no_hits = int((~predicted_hit & ~observed_hit).sum())
    matrices[horizon] = pd.DataFrame(
        [[correct_hits, false_alarms], [misses, correct_no_hits]],
        index=['Model said: hit', 'Model said: no hit'],
        columns=['Radar later: hit', 'Radar later: no hit'],
    )
    summaries.append({
        'look_ahead_minutes': horizon,
        'times_we_alerted': correct_hits + false_alarms,
        'alerts_that_were_right': correct_hits,
        'false_alarms': false_alarms,
        'actual_hits_we_missed': misses,
        'correct_no_hit_calls': correct_no_hits,
    })

summary = pd.DataFrame(summaries).set_index('look_ahead_minutes')
display(summary)

for horizon, matrix in matrices.items():
    row = summary.loc[horizon]
    print(
        f'In {int(horizon)} minutes: we alerted {int(row.times_we_alerted)} times; '
        f'{int(row.alerts_that_were_right)} alerts were right, '
        f'{int(row.false_alarms)} were false alarms, and '
        f'we missed {int(row.actual_hits_we_missed)} actual office hits.'
    )
    display(matrix)

## Calibration check

Each point groups forecasts with similar predicted probability. A well-calibrated model would place points near the diagonal: for example, forecasts around 30% should verify near 30% of the time. Sparse bins are evidence to collect more events, not evidence to tune the model.

In [ ]:
bins = np.linspace(0, 1, 11)
fig, axes = plt.subplots(1, len(sorted(results.horizon_minutes.unique())), figsize=(8 * results.horizon_minutes.nunique(), 6), squeeze=False, layout='constrained')
for axis, horizon in zip(axes[0], sorted(results.horizon_minutes.unique())):
    frame = results[results.horizon_minutes == horizon].copy()
    frame['probability_bin'] = pd.cut(frame['combined_probability'], bins, include_lowest=True)
    reliability = frame.groupby('probability_bin', observed=True).agg(
        mean_forecast=('combined_probability', 'mean'),
        observed_rate=('observed_35dbz_coverage', 'mean'),
        row_count=('event_id', 'size'),
    ).dropna()
    axis.plot([0, 1], [0, 1], linestyle='--', color='0.65', label='perfect calibration')
    size = 25 + 8 * reliability['row_count']
    axis.scatter(reliability['mean_forecast'], reliability['observed_rate'], s=size, color='#24a0ed', edgecolors='white', linewidths=1.2, label='forecast bins')
    for _, point in reliability.iterrows():
        axis.annotate(f"n={int(point.row_count)}", (point.mean_forecast, point.observed_rate), xytext=(5, 5), textcoords='offset points')
    axis.set(title=f'+{horizon} minute Kalman baseline', xlim=(0, 1), ylim=(0, 1), aspect='equal', xlabel='mean forecast probability', ylabel='observed hit rate')
    axis.legend(loc='upper left')
plt.show()